# Compare GPU Roofline Results

This notebook reloads and re-parses Nsight Compute report files for multiple GPUs, normalizes the data, and compares per-kernel roofline metrics (traffic, FLOPs, arithmetic intensity, and timing) across devices for both CUDA and OpenMP builds.

In [1]:
import pandas as pd
import os
import glob
import numpy as np
import seaborn as sns
import json
import matplotlib.pyplot as plt
import re
import sys
from tqdm import tqdm

sys.path.append('../')
from utils import *
from gatherData import _parse_ncu_report, roofline_results_to_df, calc_roofline_data

from tqdm.contrib.concurrent import process_map
from functools import partial
import os
from os import path

from sass_helper import *

/Users/gbolet/miniconda3/envs/gpuflopbench-updated/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# kernel key and metrics
key_cols = ['source', 'Kernel Name', 'exeArgs', 'Block Size', 'Grid Size', 'model_type']
device_col = 'device'

groupings = key_cols + [device_col]
metrics = ['SP_FLOP', 'DP_FLOP', 'HP_FLOP', 'INTOP', 'traffic',
           'bytesRead', 'bytesWrite', 'bytesTotal',
           'dpAI', 'spAI', 'hpAI',
           'dpPerf', 'spPerf', 'hpPerf', 'xtime']

log_metrics = ['traffic', 'xtime']

# all cols
all_cols = groupings + metrics + ['sample']

# markers we should ignore / drop kernels containing these from the dataset
library_markers = [ 'cub::', 'thrust::', '__cuda_' ]

In [3]:
df = pd.read_csv('all-NCU-GPU-Data.csv', low_memory=False)
print(df.shape)
print(df.columns)


(20044, 2659)
Index(['ID', 'Process ID', 'Process Name', 'Host Name', 'Kernel Name',
       'Context', 'Stream', 'Block Size', 'Grid Size', 'Device',
       ...
       'smsp__sass_inst_executed_op_shared.sum.pct_of_peak_sustained_elapsed',
       'smsp__sass_inst_executed_op_shared_gmma.sum',
       'smsp__sass_inst_executed_op_shared_gmma.sum.pct_of_peak_sustained_elapsed',
       'smsp__sass_inst_executed_op_tma_ld.sum',
       'smsp__sass_inst_executed_op_tma_red.sum',
       'smsp__sass_inst_executed_op_tma_st.sum', 'launch_cluster_dim_x',
       'launch_cluster_dim_z', 'launch_cluster_max_active',
       'launch_cluster_dim_y'],
      dtype='object', length=2659)


In [ ]:
sass = pd.read_csv('all-NCU-SASS-Data.csv', low_memory=False)
print(sass.shape)
print(sass.columns)
print(sass.columns[68], sass.columns[74])

In [ ]:
display(sass.head())

In [ ]:
sass_columns_to_keep = [
    'Kernel Name',
    'Address',
    'device',
    'codename',
    'exeArgs',
    'source',
    'Source',
    'sample',
    'model_type',
    'Demangled Name',
]
sass = sass[sass_columns_to_keep]

In [ ]:
sass = sass.sort_values(by = ['codename', 'Kernel Name', 'device', 'exeArgs', 'source', 'sample', 'model_type', 'Demangled Name'], ignore_index=True)

# for each column, we need to make sure it's a string and strip it
for col in sass_columns_to_keep:
    sass[col] = sass[col].astype(str).str.strip()

In [ ]:
# this elusive geglu kernel, let's see how many particualar samples of particular addresses we have
# for some strange reason, there is a whole sample that is repeated
subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['device'] == 'H100') &
              (sass['Address'].str.contains('ad400'))]

# for this subset, display which columns' are different



display(subset)

In [ ]:

subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['device'] == 'H100') &
              (sass['Address'].str.contains('ac200'))]

display(subset)

In [ ]:

display(sass.head(15))

In [ ]:
# for the sass dataframe, group it by 'Kernel Name', 'device', 'codename', 'exeArgs', and 'source'

# for some strange reason, there are duplicate rows, so we need to drop them first
sass['opcode'] = sass['Source'].apply(lambda x: extract_opcode_from_line(x))
sass['is_predicated'] = sass['Source'].apply(lambda x: detect_guard_pred_instruction(x))

print(sass.shape)
sass = sass.drop_duplicates(ignore_index=True).reset_index(drop=True)
print(sass.shape)

sass_grouped = sass.groupby(['Kernel Name', 'device', 'codename', 'exeArgs', 'source', 'sample', 'model_type', 'Demangled Name'], dropna=False)


In [ ]:

subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['device'] == 'H100') &
              (sass['Address'].str.contains('ad400'))]

display(subset)

for col in subset.columns:
    print(subset[col].value_counts())

In [ ]:

new_sasses = []

counter = 0
# for each group, let's analyze the SASS source code and create an instruction mix profile
# each group should be one particular kernel, we have 3 samples, so the output should have 3 rows per kernel
for name, group in tqdm(sass_grouped):
    group = group.sort_values(by='Address') 

    #print(group.head())
    if name[0] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_' and name[1] == 'H100':
        counter += 1
        print('found a geglu', counter)
        # print the first few instructions
        display(group.head(5))
        display(group.tail(5))

    # create a histogram of the opcodes
    opcode_counts = group['opcode'].value_counts().to_dict()

    # prefix each key with 'opcode_'
    opcode_counts = {f'opcode_{k}': v for k, v in opcode_counts.items()}

    lines_of_code = group.shape[0]

    new_row = {
        'Kernel Name': name[0],
        'device': name[1],
        'codename': name[2],
        'exeArgs': name[3],
        'source': name[4],
        'sample': name[5],
        'model_type': name[6],
        'Demangled Name': name[7],
        'lines_of_code': lines_of_code,
        **opcode_counts
    }
    new_sasses.append(new_row)

    #print(new_row)


new_sass = pd.DataFrame(new_sasses)

# print(sass_grouped.head())

In [ ]:
print(new_sass.head())

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

subset = new_sass[(new_sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_')]

display(subset)


In [ ]:
# sanity check, these opcodes should be identical across samples of the same kernel
new_sass_grouped = new_sass.groupby(['Kernel Name', 'device', 'codename', 'exeArgs', 'source', 'model_type', 'Demangled Name'], dropna=False)

condensed_imix = []

for name, group in tqdm(new_sass_grouped):
    if group.shape[0] > 1:
        group = group.sort_values(by='sample')

        mismatch_found = False

        # if the lines_of_code are different, then we have a problem
        first_loc = group.iloc[0]['lines_of_code']
        for idx, row in group.iterrows():
            if row['lines_of_code'] != first_loc:
                print("Lines of code mismatch found in group:", name)
                #display(group)
                mismatch_found = True
                break

        if mismatch_found:
            continue

    condensed_imix.append(group.iloc[0])

        # this still works, but is too verbose
        # check if all rows are identical
        #first_row = group.iloc[0]
        #first_row = first_row.drop('sample')
        #for idx, row in group.iterrows():
        #    # drop the sample index column for comparison
        #    if row.sample:
        #        row = row.drop('sample')
        #    if not row.equals(first_row):
        #        print("Mismatch found in group:", name)
        #        display(group)
        #        # print the columns that are different
        #        diff_cols = first_row[first_row != row].index.tolist()
        #        print("Different columns:", diff_cols)
        #        # print the values of the different columns
        #        for col in diff_cols:
        #            print(f"Column: {col}, idx: {idx}, Value 1: {first_row[col]}, Value 2: {row[col]}")
            #assert row.equals(first_row), "Mismatch found in group"


condensed_imix_df = pd.DataFrame(condensed_imix)

In [ ]:
# let's fix the omp kernel names 

def fix_omp_kernel_name(x):
    if '__omp_offloading' in x:
        # we need to do a regex here to extract the function name after the following 
        # possible string patterns:
        # __omp_offloading_fd01_2c7d38_
        # __omp_offloading_10305_2b800c7_
        match = re.search(r'__omp_offloading_.*?_.*?_(.+)$', x)
        if match:
            return match.group(1)
    return x
# the rest of the chars are the function name and line number -- which is the same across compilations
condensed_imix_df['Kernel Name'] = condensed_imix_df['Kernel Name'].apply(fix_omp_kernel_name)

In [ ]:
print(condensed_imix_df.shape)

print(condensed_imix_df['device'].value_counts())

In [ ]:
condensed_imix_df.to_csv('all-IMIX-Data.csv', index=False)

In [ ]:
# now let's merge the imix data with the perf counter data
merged_df = pd.merge(df, condensed_imix_df, on=['Kernel Name', 'device', 'codename', 'exeArgs', 'source', 'model_type', 'Demangled Name'], suffixes=('', '_imix'))

In [ ]:
print(merged_df.shape)

print(merged_df.head(10))

In [ ]:
merged_df.to_csv('all-NCU-GPU-Data-with-IMIX.csv', index=False)